# 07 — Subject matching: which notebook does a new upload belong in?

A document dropped on the notebooks page should land in the student's notebook for that subject, or start a new one
when it is a subject they have no notebook for yet. `ml_pipeline/subject_match.py` scores each notebook as its closest
document (TF-IDF over words and character n-grams, no model call), and joins the best one only above
`MATCH_THRESHOLD`. This notebook picks that threshold.

**Data.** The AI Flashcards dataset (`datasets/processed/flashcards.parquet`) labels each source passage with a subject
(5) and a topic (4 per subject). Passages of one topic are joined into documents of 15, so a "notebook" holds documents
on a few topics of one subject, and a new upload is a document it has not seen. Two cases, half the trials each:

- **subject exists**: the new document's subject has a notebook — it must join that one;
- **new subject**: its subject has no notebook — it must start a new one, whatever the nearest notebook scores.

A wrong join (a chemistry lecture filed under physics) is the worse mistake: it mixes subjects inside a notebook.
A missed join only costs a [Đổi] press, so the threshold leans towards precision of joins.

In [1]:
import json, random, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import numpy as np, pandas as pd
from ml_pipeline import config
from ml_pipeline.subject_match import score_notebooks

df = pd.read_parquet(config.REPO_ROOT / 'datasets' / 'processed' / 'flashcards.parquet')
passages = df.drop_duplicates('source_text')[['subject', 'topic', 'source_text']]
rng = random.Random(config.RANDOM_SEED)
docs = []
for (subject, topic), g in passages.groupby(['subject', 'topic']):
    texts = g.source_text.tolist(); rng.shuffle(texts)
    for i in range(0, len(texts) - 14, 15):
        docs.append({'subject': subject, 'topic': topic, 'text': ' '.join(texts[i:i + 15])})
docs = pd.DataFrame(docs)
print(len(docs), 'documents;', docs.groupby('subject').size().to_dict())
print('median length', int(docs.text.str.len().median()), 'characters')

157 documents; {'Biology': 32, 'Chemistry': 32, 'Computer Science': 32, 'Mathematics': 30, 'Physics': 31}
median length 2155 characters


## Trials

Each trial builds one student's notebooks (one per subject, 2 documents each, from random topics) and uploads one unseen document.

In [2]:
subjects = sorted(docs.subject.unique())
trials = []
for t in range(400):
    new_subject = t % 2 == 1
    target = rng.choice(subjects)
    notebooks, used = {}, set()
    for s in subjects:
        if new_subject and s == target:
            continue
        pick = docs[docs.subject == s].sample(2, random_state=rng.randrange(10**6))
        notebooks[s] = pick.text.tolist(); used.update(pick.index)
    upload = docs[(docs.subject == target) & ~docs.index.isin(used)].sample(1, random_state=rng.randrange(10**6)).iloc[0]
    ranked = score_notebooks(upload.text, notebooks)
    trials.append({'case': 'new subject' if new_subject else 'subject exists', 'target': target,
                   'best': ranked[0][0], 'score': ranked[0][1],
                   'runner_up': ranked[1][1] if len(ranked) > 1 else 0.0})
trials = pd.DataFrame(trials)
exists = trials[trials.case == 'subject exists']
print('subject exists: nearest notebook is the right one in', f"{(exists.best == exists.target).mean():.1%}", 'of trials')
trials.groupby('case').score.describe()[['mean', 'min', '25%', '50%', '75%', 'max']].round(3)

subject exists: nearest notebook is the right one in 100.0% of trials


,mean,min,25%,50%,75%,max
case,,,,,,
new subject,0.239,0.180,0.214,0.234,0.264,0.316
subject exists,0.568,0.321,0.379,0.434,0.821,0.888


## Threshold sweep

In [3]:
rows = []
for th in np.round(np.arange(0.05, 0.61, 0.025), 3):
    joins = trials[trials.score >= th]
    right = ((joins.case == 'subject exists') & (joins.best == joins.target)).sum()
    wrong = len(joins) - right
    ok_new = ((trials.case == 'new subject') & (trials.score < th)).sum()
    rows.append({'threshold': th,
                 'decision accuracy': (right + ok_new) / len(trials),
                 'join precision': right / len(joins) if len(joins) else 1.0,
                 'joins found': right / (trials.case == 'subject exists').sum(),
                 'wrong joins': wrong})
sweep = pd.DataFrame(rows)
sweep.round(3)

,threshold,decision accuracy,join precision,joins found,wrong joins
0,0.050,0.500,0.500,1.000,200
1,0.075,0.500,0.500,1.000,200
2,0.100,0.500,0.500,1.000,200
3,0.125,0.500,0.500,1.000,200
4,0.150,0.500,0.500,1.000,200
5,0.175,0.500,0.500,1.000,200
6,0.200,0.548,0.525,1.000,181
7,0.225,0.708,0.631,1.000,117
8,0.250,0.812,0.727,1.000,75
9,0.275,0.912,0.851,1.000,35


In [4]:
best = sweep.sort_values(['decision accuracy', 'join precision'], ascending=False).iloc[0]
safe = sweep[sweep['join precision'] >= 0.98].sort_values('decision accuracy', ascending=False).iloc[0]
print('most accurate:', best.to_dict())
print('accurate with >=98% join precision:', safe.to_dict())

most accurate: {'threshold': 0.325, 'decision accuracy': 0.995, 'join precision': 1.0, 'joins found': 0.99, 'wrong joins': 0.0}
accurate with >=98% join precision: {'threshold': 0.325, 'decision accuracy': 0.995, 'join precision': 1.0, 'joins found': 0.99, 'wrong joins': 0.0}


## The student's own documents

Three real documents from the dev account (dumped to `outputs/real_docs_dev.json`; their own content, so not committed): two on the CityHelp app project, one on AI ethics in education.

In [5]:
real = json.load(open(Path.cwd().resolve().parents[0] / 'outputs' / 'real_docs_dev.json'))
for doc in real:
    others = {f"{d['notebook']} / {d['title']}": [d['text']] for d in real if d['id'] != doc['id']}
    print(doc['title'][:30].ljust(30), [(k, round(v, 3)) for k, v in score_notebooks(doc['text'], others)])

CityHelp_Consolidated          [('testings / CityHelp', 0.615), ('testings / Education for AI Ethics', 0.273)]
Education for AI Ethics        [('ád / CityHelp_Consolidated', 0.273), ('testings / CityHelp', 0.265)]
CityHelp                       [('ád / CityHelp_Consolidated', 0.615), ('testings / Education for AI Ethics', 0.265)]


## Decision

**0.35.** The two cases barely overlap: a new subject's nearest notebook never scored above 0.316, and an upload whose
subject has a notebook never scored below 0.321 — and when it had one, the nearest notebook was the right one in every
trial. 0.325 is the most accurate cut (99.5%), but it sits on that seam. The student's real documents are ten times
longer than these and share more academic vocabulary across subjects (AI ethics vs. the CityHelp app: 0.27), so the
cut moves up to 0.35. On the trials that is zero wrong joins and 88% of joins still found. A missed join costs a
[Đổi] press; a wrong one mixes two subjects in one notebook.

Limits: the dataset is English and natural-science only, with short documents. The real check is three documents,
two of which are versions of one project (0.615, joined correctly). Worth re-running once more real uploads exist.